# TP1 : déployer et valider le cluster KRaft

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 11:05-11:45, après la démo du TP (11:00-11:05) |
| Compétences | C2 KRaft et quorum ; C3 Déployer, démarrer et valider un cluster |
| Slides de rappel | S18 (installation), S19 (`server.properties`), S24 (architecture du lab) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande** à exécuter tel quel, une **zone `# TODO`** à compléter en Python, puis une **vérification** (une instruction `assert`) qui affiche `OK` ou s'arrête sur `AssertionError: A revoir : ...` avec la raison.
- Une ligne qui commence par `!` est envoyée au terminal, comme dans un terminal ouvert sur le poste. `!docker exec kafka-tools kafka-topics.sh ...` lance la commande dans le conteneur d'administration (`# [tools]` dans les énoncés Markdown) ; `!docker compose ...` s'exécute dans le dossier `lab/` (`# [hôte]`). Dans une ligne `!`, `{expression}` insère une valeur Python, et les accolades littérales se doublent : `{{{{.Name}}}}` donne `{{.Name}}` à Docker. `sortie = !commande` range les lignes de la sortie dans une liste ; `sortie.n` les joint en un seul texte.
- Le client Python est `confluent-kafka` (la bibliothèque librdkafka). `AdminClient(CONF)` crée un client d'administration connecté au listener INTERNAL (`BS`). Le TP crée un client neuf par cellule : un client qui a déjà servi garde en cache le coordinateur des groupes, et échoue quand ce coordinateur change de broker.
- Les points 7 (étapes), 8 (commandes), 9 (résultats attendus) et 10 (vérification) du format de TP sont regroupés étape par étape.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
KAFKA_UI = "http://kafka-ui:8080"
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, ecrites ici : elles ne font pas partie de confluent-kafka
def attendre(condition, timeout=60, pas=2, message="condition"):
    """Rappelle condition() jusqu'a obtenir une valeur vraie, et la renvoie."""
    debut = time.time()
    while True:
        try:
            valeur = condition()
        except Exception:
            valeur = None
        if valeur:
            return valeur
        if time.time() - debut >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(pas)

def tableau(lignes, entetes):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in ligne) + " |" for ligne in lignes]
    display(Markdown("\n".join(md)))

In [ ]:
import re
import time
from collections import Counter

import requests

print("Bootstrap INTERNAL :", BS)

## 1. Objectifs

À la fin du TP, vous savez :

- vérifier que les 3 nœuds du lab sont sains ;
- lire dans `server.properties` ce qui fait d'un nœud un broker et un contrôleur KRaft ;
- vérifier qu'un nœud a été formaté et à quel cluster il appartient ;
- identifier le contrôleur actif et l'état de réplication du quorum ;
- décrire le cluster avec le client Python, et savoir ce que ce client ne voit pas ;
- retrouver un événement dans les traces d'un nœud ;
- lire le journal de métadonnées `__cluster_metadata` ;
- constater qu'un redémarrage du contrôleur actif provoque une nouvelle élection sans interruption.

## 2. Durée

40 minutes, après 5 minutes de démonstration.

| Étape | Durée |
|---|---|
| 7.1 Vérifier les conteneurs | 3 min |
| 7.2 Lire la configuration KRaft d'un nœud | 5 min |
| 7.3 Vérifier le formatage du stockage | 4 min |
| 7.4 Interroger le quorum | 5 min |
| 7.5 Décrire le cluster en Python | 6 min |
| 7.6 Traces : retrouver l'élection du contrôleur (défi) | 5 min |
| 7.7 Journal de métadonnées | 4 min |
| 7.8 Kafka UI et son API | 2 min |
| 7.9 Redémarrer le contrôleur actif | 6 min |

## 3. Prérequis

- Lab démarré dans un terminal, dans le dossier `lab/` : `scripts/start.sh extended` (ou `core` si votre poste a moins de 16 Go de RAM), puis `scripts/validate.sh`, qui doit afficher « Environnement valide. ».
- JupyterLab ouvert sur http://localhost:8888, avec ce notebook.
- Notions : slides S4 à S24 (topics, partitions, brokers, KRaft, listeners).

## 4. Architecture utilisée

```
                         Réseau Docker kafka-lab
  +----------------------------------------------------------------------+
  |   kafka-1 (az-a)          kafka-2 (az-b)          kafka-3 (az-c)     |
  |   broker + contrôleur     broker + contrôleur     broker + contrôleur |
  |        \                       |                       /             |
  |         +------ quorum KRaft : 3 votants (port 9093) -+              |
  |                                                                      |
  |   jupyter : ce notebook (client Python, docker compose)              |
  |   tools   : CLI Kafka (kafka-*.sh, $BS), appelée par docker exec    |
  |   kafka-ui : http://localhost:8080                                   |
  |   [extended] prometheus, grafana, kafka-exporter, loadgen            |
  +----------------------------------------------------------------------+
```

| Listener | Port | Protocole | Usage dans ce TP |
|---|---|---|---|
| CONTROLLER | 9093 | PLAINTEXT | Quorum KRaft, entre contrôleurs |
| INTERNAL | 19092 | PLAINTEXT | CLI de `tools` et client Python du notebook (`BS`) |
| EXTERNAL | 9092, 9192, 9292 | PLAINTEXT | Clients lancés sur l'hôte (non utilisé ici) |
| SECURE | 19094 | SASL_SSL | TP5 |

## 5. Contexte métier

NovaShop, site de commerce en ligne, migre ses flux de commandes, de paiements et de navigation vers Kafka 4.3. L'équipe plateforme, dont vous faites partie, reçoit le nouveau cluster de recette : 3 nœuds en mode KRaft, sans ZooKeeper.

Avant d'ouvrir le cluster aux équipes de développement, vous devez produire un procès-verbal de recette technique. Il doit prouver que le cluster est sain, que le quorum fonctionne, que les traces sont exploitables et qu'un redémarrage de nœud ne l'interrompt pas. Ce notebook le construit au fil des étapes : il s'affiche en fin de TP.

## 6. Énoncé

Collectez les preuves suivantes :

1. l'identifiant du cluster et le fait que chaque nœud a été formaté avec cet identifiant ;
2. l'identité du contrôleur actif et l'état de réplication du quorum ;
3. la liste des brokers enregistrés et la version de métadonnées en vigueur ;
4. la trace qui prouve l'élection du contrôleur actif ;
5. le comportement du cluster quand le contrôleur actif redémarre.

## 7. Étapes

### 7.1 Vérifier les conteneurs

Le lab a été démarré dans un terminal. On vérifie ici, depuis le notebook, que les 3 nœuds Kafka sont sains : leur healthcheck teste que le listener INTERNAL accepte les connexions.

**Exemple en ligne de commande** (équivalent de `docker compose ps` dans `lab/`) :

In [ ]:
!docker compose ps --format 'table {{{{.Name}}}}\t{{{{.Status}}}}'

In [ ]:
# TODO 7.1 : lister les noeuds Kafka sains
# La commande ci-dessous renvoie une ligne "nom etat_de_sante" par conteneur.
lignes = !docker compose ps --format '{{{{.Name}}}} {{{{.Health}}}}' 2>&1

# Construisez la liste noeuds_sains : noms des conteneurs qui commencent par "kafka-"
# et dont l'etat de sante vaut "healthy".
# Indice : "kafka-1 healthy".partition(" ") renvoie ("kafka-1", " ", "healthy").
noeuds_sains = []  # a completer

print(noeuds_sains)

In [ ]:
# Verification 7.1
assert sorted(noeuds_sains) == ["kafka-1", "kafka-2", "kafka-3"], "A revoir : 3 noeuds Kafka sains : %s" % noeuds_sains
print("OK  3 noeuds Kafka sains : %s" % noeuds_sains)

**Résultat attendu** : `kafka-1`, `kafka-2` et `kafka-3` en `Up ... (healthy)`, ainsi que `jupyter`, `kafka-tools` et `kafka-ui`. En mode `extended` s'ajoutent `prometheus`, `grafana`, `kafka-exporter`, `loadgen-producer` et `loadgen-consumer`.

### 7.2 Lire la configuration KRaft d'un nœud

Le fichier `server.properties` de chaque nœud est dans `lab/config/kafka/kafka-N/`. Il dit quel rôle joue le nœud (`process.roles`), son identifiant, comment joindre le quorum, où il écoute et où il range ses données.

**Exemple en ligne de commande** :

In [ ]:
!grep -E '^(process.roles|node.id|controller.quorum.bootstrap.servers|controller.listener.names|listeners|advertised.listeners|log.dirs)=' config/kafka/kafka-1/server.properties

In [ ]:
# TODO 7.2 : ecrire la fonction proprietes(texte, sep)
# Elle recoit le contenu d'un fichier de proprietes et renvoie un dictionnaire {cle: valeur}.
#   - ignorer les lignes vides et les commentaires (qui commencent par #) ;
#   - ignorer les lignes qui ne contiennent pas le separateur ;
#   - couper chaque ligne au PREMIER separateur seulement : une valeur peut contenir "=" ;
#   - retirer les espaces autour de la cle et de la valeur (methode strip()).
def proprietes(texte, sep="="):
    resultat = {}
    # a completer
    return resultat


# Le dossier lab/ est monte dans le conteneur du notebook : lecture directe du fichier
with open("config/kafka/kafka-1/server.properties") as f:
    conf1 = proprietes(f.read())

print(conf1.get("process.roles"), "- node.id", conf1.get("node.id"))

In [ ]:
# Verification 7.2
assert proprietes("a = 1=2\n# b=3\n\nc") == {"a": "1=2"}, "A revoir : proprietes() : commentaire, ligne vide et '=' dans la valeur"
print("OK  proprietes() : commentaire, ligne vide et '=' dans la valeur")
assert conf1.get("process.roles") == "broker,controller", "A revoir : kafka-1 est a la fois broker et controleur"
print("OK  kafka-1 est a la fois broker et controleur")
assert conf1.get("controller.quorum.bootstrap.servers", "").count(":9093") == 3, "A revoir : 3 adresses de contact du quorum"
print("OK  3 adresses de contact du quorum")

# Les 3 noeuds cote a cote
lignes_conf = []
for n in (1, 2, 3):
    with open("config/kafka/kafka-%d/server.properties" % n) as f:
        c = proprietes(f.read())
    lignes_conf.append((n, c.get("node.id"), c.get("process.roles"), c.get("broker.rack"), c.get("log.dirs")))
tableau(lignes_conf, ["noeud", "node.id", "process.roles", "broker.rack", "log.dirs"])

**Résultat attendu** pour `kafka-1` :

```
process.roles=broker,controller
node.id=1
controller.quorum.bootstrap.servers=kafka-1:9093,kafka-2:9093,kafka-3:9093
controller.listener.names=CONTROLLER
listeners=CONTROLLER://:9093,INTERNAL://:19092,EXTERNAL://:9092,SECURE://:19094
advertised.listeners=CONTROLLER://kafka-1:9093,INTERNAL://kafka-1:19092,EXTERNAL://localhost:9092,SECURE://kafka-1:19094
log.dirs=/var/lib/kafka/data
```

### 7.3 Vérifier le formatage du stockage

En KRaft, un nœud ne démarre que si son répertoire de données a été formaté avec `kafka-storage.sh format`. Le formatage écrit `meta.properties`, qui fixe l'identité du nœud : `cluster.id`, `node.id` et `directory.id`.

**Exemple en ligne de commande** :

In [ ]:
!docker compose exec -T kafka-1 /opt/kafka/bin/kafka-storage.sh info -c /opt/lab/node/server.properties
!docker compose logs kafka-1 | grep -E 'start-node|Formatting|already formatted|Kafka Server started'

In [ ]:
# TODO 7.3 : relever le cluster.id des 3 noeuds
# Pour chaque noeud n de 1 a 3 :
#   - lisez le fichier /var/lib/kafka/data/meta.properties du conteneur kafka-n :
#       texte = !docker compose exec -T kafka-{n} cat /var/lib/kafka/data/meta.properties
#   - transformez-le en dictionnaire avec proprietes() ;
#   - rangez son cluster.id dans le dictionnaire ids, avec n pour cle.
ids = {}

print(ids)

In [ ]:
# Verification 7.3
assert len(ids) == 3, "A revoir : meta.properties lu sur les 3 noeuds"
print("OK  meta.properties lu sur les 3 noeuds")
assert len(set(ids.values())) == 1, "A revoir : un seul cluster.id pour les 3 noeuds : %s" % set(ids.values())
print("OK  un seul cluster.id pour les 3 noeuds : %s" % set(ids.values()))

**Résultat attendu** : `Found metadata: {cluster.id=W5FmqobtRp-BMcXlZhVKBw, directory.id=4EubdzSZTbykbeqlrbgz1Q, node.id=1, version=1}`. Au premier démarrage, les traces contiennent `Formatting dynamic metadata voter directory /var/lib/kafka/data with metadata.version 4.3-IV0.` ; aux démarrages suivants, `All of the log directories are already formatted.`

### 7.4 Interroger le quorum

Le quorum KRaft élit un leader parmi les contrôleurs votants : c'est le contrôleur actif, seul à écrire dans le journal de métadonnées. L'outil `kafka-metadata-quorum.sh` interroge le quorum ; le client Python n'a pas d'équivalent.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication

In [ ]:
# TODO 7.4 : ecrire la fonction etat_quorum()
# Elle doit :
#   - lancer "kafka-metadata-quorum.sh --bootstrap-server $BS describe --status" dans une cellule : sortie = !docker exec kafka-tools ... ;
#   - joindre les lignes en un texte avec sortie.n, puis le transformer en dictionnaire avec proprietes(texte, sep=":") ;
#   - renvoyer le couple (LeaderId, LeaderEpoch), convertis en entiers avec int().
def etat_quorum():
    # a completer
    return None, None


leader, epoque = etat_quorum()
print("Controleur actif :", leader, "- epoque :", epoque)

In [ ]:
# Verification 7.4
assert leader in (1, 2, 3), "A revoir : controleur actif identifie : %s" % leader
print("OK  controleur actif identifie : %s" % leader)
assert isinstance(epoque, int) and epoque >= 1, "A revoir : epoque du leader : %s" % epoque
print("OK  epoque du leader : %s" % epoque)

**Résultat attendu** : `LeaderId` vaut 1, 2 ou 3 ; `LeaderEpoch` vaut 1 sur un lab qui vient de démarrer. `CurrentVoters` liste les 3 nœuds et `CurrentObservers` est vide. `describe --replication` affiche 1 ligne `Leader` et 2 lignes `Follower`, avec un `Lag` de 0 ou proche de 0.

### 7.5 Décrire le cluster en Python

Les applications et les outils d'administration voient le cluster à travers les brokers. Le client Python obtient l'identifiant du cluster et la liste des brokers par l'API `describe_cluster()`, équivalente de `kafka-cluster.sh` et `kafka-broker-api-versions.sh`.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools bash -c "kafka-broker-api-versions.sh --bootstrap-server $BS | grep '^kafka'"
!docker exec kafka-tools kafka-cluster.sh cluster-id --bootstrap-server $BS
!docker exec kafka-tools kafka-features.sh --bootstrap-server $BS describe

In [ ]:
# TODO 7.5 : decrire le cluster avec le client Python
# admin.describe_cluster() renvoie un "futur" : .result() attend la reponse du cluster.
# Le resultat a trois attributs : cluster_id, controller (un noeud) et nodes (liste de noeuds).
# Chaque noeud a les attributs id, host, port et rack.
#   1. Rangez le resultat dans la variable cluster.
#   2. Construisez brokers : liste de tuples (id, host, rack), triee par id (fonction sorted).
cluster = None
brokers = []  # a completer

print(brokers)

In [ ]:
# Verification 7.5
assert cluster is not None and cluster.cluster_id == ids.get(1), "A revoir : meme identifiant de cluster qu'a l'etape 7.3"
print("OK  meme identifiant de cluster qu'a l'etape 7.3")
assert [b[2] for b in brokers] == ["az-a", "az-b", "az-c"], "A revoir : 3 brokers, un par rack : %s" % brokers
print("OK  3 brokers, un par rack : %s" % brokers)

Le résultat de `describe_cluster()` contient aussi un champ `controller`. Comparez-le, sur 5 appels, au leader du quorum lu à l'étape 7.4. Qu'observez-vous ? Notez votre explication avant de lire la correction.

In [ ]:
admin = AdminClient(CONF)
for i in range(5):
    print("describe_cluster().controller :", admin.describe_cluster().result().controller.id,
          "| leader du quorum :", etat_quorum()[0])

**Résultat attendu** : 3 brokers `kafka-1`, `kafka-2`, `kafka-3` dans les racks `az-a`, `az-b`, `az-c`. `metadata.version` est finalisée à `4.3-IV0` et `kraft.version` à 1.

### 7.6 Défi : retrouver l'élection du contrôleur dans les traces

Chaque nœud écrit ses traces dans `/opt/kafka/logs`. Le nœud qui devient contrôleur actif écrit la ligne `Becoming the active controller` dans `controller.log`.

**Exemple en ligne de commande** :

In [ ]:
!docker compose exec -T kafka-1 ls /opt/kafka/logs

In [ ]:
# TODO 7.6 (defi) : trouver le noeud qui porte la trace d'election
# Pour chaque noeud n de 1 a 3, cherchez la ligne "Becoming the active controller"
# dans ses fichiers de traces du controleur, puis rangez dans elections[n]
# la liste des lignes trouvees (liste vide si aucune).
# Commande de depart pour kafka-1, incomplete : elle ne lit qu'un seul fichier.
#   docker compose exec -T kafka-1 grep -h "Becoming the active controller" /opt/kafka/logs/controller.log
# Indice : relisez la liste des fichiers ci-dessus, et pensez a l'endroit ou l'etoile est developpee.
elections = {}

for n, trouvees in elections.items():
    print("kafka-%d : %d ligne(s)" % (n, len(trouvees)))

In [ ]:
# Verification 7.6
assert len(elections) == 3, "A revoir : traces des 3 noeuds examinees"
print("OK  traces des 3 noeuds examinees")
assert leader in [n for n, trouvees in elections.items() if trouvees], "A revoir : le controleur actif (%s) porte une trace d'election" % leader
print("OK  le controleur actif (%s) porte une trace d'election" % leader)

**Résultat attendu** : sur un lab qui vient de démarrer, une seule ligne, sur le nœud élu leader : `Becoming the active controller at epoch 1`. Un nœud qui a déjà été contrôleur actif après un redémarrage porte aussi des lignes, avec des époques croissantes.

### 7.7 Explorer le journal de métadonnées

En KRaft, les métadonnées (topics, partitions, brokers, configurations) sont un journal Kafka : la partition `__cluster_metadata-0`, répliquée entre les contrôleurs. `kafka-dump-log.sh --cluster-metadata-decoder` décode ses enregistrements.

**Exemple en ligne de commande** :

In [ ]:
!docker compose exec -T kafka-1 ls /var/lib/kafka/data/__cluster_metadata-0
# Nombre d'enregistrements par type
!docker compose exec -T kafka-1 sh -c '/opt/kafka/bin/kafka-dump-log.sh --cluster-metadata-decoder --files /var/lib/kafka/data/__cluster_metadata-0/00000000000000000000.log 2>/dev/null | grep -o "\"type\":\"[A-Z_]*\"" | sort | uniq -c | sort -rn'

In [ ]:
# TODO 7.7 : compter les types d'enregistrements en Python
# La sortie complete du decodage est longue : elle n'est pas affichee.
dump = !docker compose exec -T kafka-1 sh -c '/opt/kafka/bin/kafka-dump-log.sh --cluster-metadata-decoder --files /var/lib/kafka/data/__cluster_metadata-0/00000000000000000000.log 2>/dev/null' 2>&1
dump = dump.n

# 1. Trouvez tous les types avec re.findall(r'"type":"([A-Z_]+)"', dump),
#    comptez-les avec Counter et rangez le resultat dans types.
# 2. Rangez dans topics_crees les noms des topics crees (enregistrements TOPIC_RECORD) :
#    re.findall(r'"type":"TOPIC_RECORD".*?"name":"([^"]+)"', dump)
types = Counter()
topics_crees = []

print(types.most_common(5))
print(topics_crees)

In [ ]:
# Verification 7.7
assert "__consumer_offsets" in topics_crees, "A revoir : creation de __consumer_offsets retrouvee"
print("OK  creation de __consumer_offsets retrouvee")
assert types["TOPIC_RECORD"] == len(topics_crees), "A revoir : un TOPIC_RECORD par topic cree (%d)" % len(topics_crees)
print("OK  un TOPIC_RECORD par topic cree (%d)" % len(topics_crees))
assert types["PARTITION_RECORD"] >= 50, "A revoir : " + ("%d PARTITION_RECORD (50 pour __consumer_offsets, plus les autres topics)"
         % types["PARTITION_RECORD"])
print("OK  " + ("%d PARTITION_RECORD (50 pour __consumer_offsets, plus les autres topics)"
         % types["PARTITION_RECORD"]))

**Résultat attendu** : un segment `00000000000000000000.log`, un fichier `quorum-state` et un instantané `.checkpoint`. Les types les plus nombreux sont `NO_OP_RECORD` et `PARTITION_RECORD`. On trouve un `TOPIC_RECORD` par topic : `__consumer_offsets`, plus `loadgen` et `lab-validate` en mode `extended`.

### 7.8 Kafka UI et son API

Ouvrez http://localhost:8080, puis les menus « Brokers » et « Topics » : nombre de brokers, contrôleur, liste des topics. L'interface expose aussi une API HTTP, joignable depuis le notebook à l'adresse `KAFKA_UI` (`http://kafka-ui:8080`).

In [ ]:
# TODO 7.8 : interroger l'API de Kafka UI
# requests.get(KAFKA_UI + "/api/clusters", timeout=10).json() renvoie une liste :
# un element (dictionnaire) par cluster declare dans l'interface.
# Rangez le premier element dans ui.
ui = {}

print(ui.get("name"), ui.get("status"), ui.get("brokerCount"))

In [ ]:
# Verification 7.8
assert ui.get("status") == "ONLINE", "A revoir : cluster %s ONLINE dans Kafka UI" % ui.get("name")
print("OK  cluster %s ONLINE dans Kafka UI" % ui.get("name"))
assert ui.get("brokerCount") == 3, "A revoir : 3 brokers vus par Kafka UI"
print("OK  3 brokers vus par Kafka UI")

### 7.9 Redémarrer le contrôleur actif

Un arrêt contrôlé du contrôleur actif déclenche une nouvelle élection parmi les votants restants. Le nœud redémarré rejoint ensuite le quorum comme follower. Le trafic `loadgen` continue pendant toute l'opération : regardez Grafana si le mode `extended` tourne.

**Équivalent en ligne de commande** (ne pas lancer en plus du TODO) :

```bash
# [hôte] remplacer N par le LeaderId
docker compose restart kafka-N
docker compose exec tools kafka-metadata-quorum.sh --bootstrap-server kafka-1:19092,kafka-2:19092,kafka-3:19092 describe --status
docker compose ps kafka-N
```

In [ ]:
# TODO 7.9 : redemarrer le controleur actif et mesurer la bascule
#   1. Relevez (avant, epoque_avant) avec etat_quorum().
#   2. Notez l'heure de depart : debut = time.time().
#   3. Redemarrez le noeud : `!docker compose restart kafka-{avant}`.
#   4. Ecrivez une fonction nouveau_leader() qui renvoie etat_quorum() si le leader
#      a change, et None sinon. Passez-la a attendre(nouveau_leader, timeout=60)
#      pour obtenir (apres, epoque_apres).
#   5. Calculez duree = time.time() - debut.
avant, epoque_avant = None, None
apres, epoque_apres = None, None
duree = None

print("Leader avant :", avant, "- apres :", apres, "- duree : %s s" % duree)

In [ ]:
# Verification 7.9
assert apres not in (None, avant), "A revoir : nouveau leader du quorum : %s (avant : %s)" % (apres, avant)
print("OK  nouveau leader du quorum : %s (avant : %s)" % (apres, avant))
assert epoque_apres is not None and epoque_apres > epoque_avant, "A revoir : epoque passee de %s a %s" % (epoque_avant, epoque_apres)
print("OK  epoque passee de %s a %s" % (epoque_avant, epoque_apres))

# Retour du noeud redemarre : healthy en 15 a 20 s, puis follower sans retard
def sante(n):
    sortie = !docker compose ps --format '{{{{.Health}}}}' kafka-{n} 2>&1
    return sortie.n


attendre(lambda: sante(avant) == "healthy", timeout=120, message="kafka-%d healthy" % avant)
!docker compose logs kafka-{avant} --since 2m | grep -E 'start-node|already formatted|Kafka Server started'
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication

**Résultat attendu** :

- le redémarrage dure quelques secondes ;
- immédiatement après, `LeaderId` désigne un autre nœud et `LeaderEpoch` a augmenté de 1 ;
- le nœud redémarré revient `healthy` en 15 à 20 s ; ses traces affichent `All of the log directories are already formatted.` puis `Kafka Server started` ;
- `describe --replication` le montre de nouveau `Follower` avec un `Lag` de 0.

## Procès-verbal de recette

La cellule suivante rassemble les preuves collectées. Une ligne « à compléter » renvoie à l'étape dont le `TODO` n'est pas terminé.

In [ ]:
def valeur(v, defaut="a completer"):
    return defaut if v in (None, {}, [], "") else v


features = !docker exec kafka-tools kafka-features.sh --bootstrap-server $BS describe 2>&1
features = features.n
version_meta = re.search(r"metadata\.version.*?FinalizedVersionLevel:\s*(\S+)", features)

tableau([
    ("Identifiant du cluster", valeur(cluster.cluster_id if cluster else None), "describe_cluster(), 7.5"),
    ("Noeuds formates avec cet identifiant", valeur("%d/3" % len(ids) if ids else None), "meta.properties, 7.3"),
    ("Controleur actif et epoque", valeur("%s, epoque %s" % (leader, epoque) if leader else None), "describe --status, 7.4"),
    ("Brokers enregistres", valeur(", ".join("%s (%s)" % (b[1], b[2]) for b in brokers)), "describe_cluster(), 7.5"),
    ("metadata.version", version_meta.group(1) if version_meta else "?", "kafka-features.sh describe"),
    ("Noeud portant la trace d'election", valeur(", ".join("kafka-%d" % n for n, t in elections.items() if t)), "controller.log*, 7.6"),
    ("Nouveau controleur apres redemarrage",
     valeur("%s, epoque %s, en %s s" % (apres, epoque_apres, duree) if apres else None), "7.9"),
], ["Preuve", "Valeur relevee", "Source"])

## 11. Questions de réflexion

1. Pourquoi le quorum compte-t-il 3 votants plutôt que 2 ou 4 ? Combien de pannes de contrôleur chaque taille tolère-t-elle ?
2. Pourquoi `advertised.listeners` diffère-t-il de `listeners` ? Que se passerait-il si un nœud annonçait un nom d'hôte inexistant ?
3. Où la liste des topics est-elle stockée en mode KRaft ? Qui l'écrit ?
4. Au redémarrage du contrôleur actif, les producteurs et consommateurs sont-ils interrompus ? Pourquoi ?
5. Dans ce lab, chaque nœud est à la fois broker et contrôleur. Quels risques ce choix présente-t-il en production ?
6. Pourquoi le formatage du stockage est-il une étape explicite en KRaft ?
7. Pourquoi le champ `controller` de `describe_cluster()` ne suffit-il pas à superviser le contrôleur actif ?

## 12. Erreurs fréquentes

| Symptôme | Cause | Correction |
|---|---|---|
| `NameError: name 'proprietes' is not defined` | Cellule de la fonction non exécutée, ou noyau redémarré | Relancer les cellules précédentes (*Run > Run All Above*) |
| `AssertionError: A revoir : ...` | Le `TODO` de l'étape n'est pas terminé ou donne un autre résultat | Relire la consigne du `TODO`, afficher la variable concernée |
| `KafkaException ... _DESTROY` | `AdminClient` créé puis détruit avant la réponse | Garder le client dans une variable (`admin = AdminClient(CONF)`) jusqu'à la fin de la requête |
| `KafkaException ... _TRANSPORT` ou délai dépassé | Lab arrêté ou nœud en panne | `scripts/validate.sh` dans un terminal |
| `grep` ne trouve pas `Becoming the active controller` | Recherche dans `controller.log` seul, alors que la ligne est dans un fichier déjà tourné, ou sur un nœud non élu | Chercher dans `controller.log*`, avec `sh -c`, sur les 3 nœuds |
| `cd`/`FileNotFoundError` sur `config/...` ou `FORMATION_DIR` introuvable | Lab démarré par `docker compose up` : le conteneur du notebook ne connaît pas le chemin de la formation sur l'hôte | Relancer le lab avec `scripts/start.sh` |
| Un nœud reste `health: starting` | Mémoire Docker insuffisante | Passer en mode `core` ; `docker compose logs kafka-N --tail 50` |

## 13. Solution formateur

Voir `corrige.ipynb` (et `corrige.md` pour la version en ligne de commande).

## 14. Variante avancée

Pour les participants en avance :

1. Mesurez le temps nécessaire pour que le quorum élise un nouveau leader quand le contrôleur actif est arrêté brutalement (`docker compose kill kafka-N`) au lieu d'être redémarré proprement. Réutilisez `etat_quorum()` et `attendre()`. Redémarrez ensuite le nœud avec `docker compose start kafka-N`.
2. Écrivez une fonction `retard_followers()` qui lit `describe --replication` et renvoie le `Lag` de chaque nœud, puis appelez-la pendant le redémarrage d'un follower.
3. Cherchez dans `kafka-features.sh --help` la commande qui afficherait les niveaux de fonctionnalités associés à une version de Kafka (`version-mapping`), et lancez-la pour la version 4.3.